## Imports

In [ ]:
import os
import time
import pathlib as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

# módulos do seu projeto
from knn import set_train_test, knn_predict, plot_diagram  # OK: já existem
# from gan_testes import run_experiment
from gan import run_experiment  # <- troque para gan_testes se for o caso

ROOT = pl.Path(".").resolve()
DATA_LABELED = ROOT / "data" / "Labeled"
EXTRA = ROOT / "extra"

# figuras/relatórios sairão aqui
RUN_DIR = ROOT / "results" / time.strftime("run_%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

def load_all():
    """Carrega CSVs rotulados e curvas teóricas (iguais às do knn.py)."""
    df_H1 = pd.read_csv(DATA_LABELED / "H1_labeled.csv")
    df_H2 = pd.read_csv(DATA_LABELED / "H2_labeled.csv")
    df_H3 = pd.read_csv(DATA_LABELED / "H3_labeled.csv")

    df_H1_theory = pd.read_csv(EXTRA / "pontos_plot_XXZ.csv", header=None)
    df_H2_theory = pd.read_csv(EXTRA / "data_paper_ferro_bond.csv", header=None)
    df_H3_theory = pd.DataFrame({0: [0.25, 0.5, 1.25, 1.75], 1: [0, 0, 0, 0]})
    theoretical_list = [df_H1_theory, df_H2_theory, df_H3_theory]

    return [df_H1, df_H2, df_H3], theoretical_list

def avalia_knn_plot(X_tr, y_tr, X_te, y_te, params_te, k, h_idx, titulo, theoretical_list, savepath_png):
    """Treina, avalia e plota usando suas funções; salva PNG e retorna métricas."""
    y_pred = knn_predict(X_tr, y_tr, X_te, k)

    acc = accuracy_score(y_te, y_pred)
    rep = classification_report(y_te, y_pred, zero_division=0)

    # plot
    plt.figure(figsize=(6,5))
    plot_diagram(params_te, y_pred, h_idx, theoretical_list)
    plt.title(f"{titulo} — K={k}")
    plt.tight_layout()
    plt.savefig(savepath_png, dpi=160, bbox_inches="tight")
    plt.close()

    return acc, rep


In [ ]:
def run_one(H_list, theoretical_list, *, H_idx_test: int, K: int = 50,
            gan_cfg: dict | None = None, samples_per_label: int = 500):
    """
    Executa baseline e com cGAN para um H_idx de teste.
    - Usa seu set_train_test() → chama prep_data() com balanceamento + spatial sign (Normalizer l2),
      depois filtra rótulos inexistentes no treino (consistente com seu código).   
    """
    tag = f"H{H_idx_test}"
    out_dir = RUN_DIR / tag
    out_dir.mkdir(exist_ok=True)

    # 1) Split e pré-processamento (seu pipeline)
    X_tr, y_tr, X_te, y_te, params_te = set_train_test(H_list, H_idx_test)

    # 2) Baseline (sem GAN)
    acc_b, rep_b = avalia_knn_plot(
        X_tr, y_tr, X_te, y_te, params_te, K, H_idx_test,
        f"Baseline (sem GAN) — {tag}", theoretical_list,
        out_dir / f"{tag}_baseline_k{K}.png"
    )

    # 3) Data augmentation com cGAN
    if gan_cfg is None:
        gan_cfg = dict(EPOCHS=100, BATCH_SIZE=80, LATENT_DIM=64, LR=2e-4, SEED=42)
    gan_cfg = {**gan_cfg, "SAMPLES_PER_LABEL": samples_per_label}

    gan_out = run_experiment(gan_cfg, X_tr, y_tr, save_csv=True)  # retorna X_aug,y_aug + checkpoints
    X_aug, y_aug = gan_out["X_aug"], gan_out["y_aug"]

    # 4) Avaliação com aumento
    acc_g, rep_g = avalia_knn_plot(
        X_aug, y_aug, X_te, y_te, params_te, K, H_idx_test,
        f"Com Data Augmentation (cGAN) — {tag}", theoretical_list,
        out_dir / f"{tag}_cgan_k{K}.png"
    )

    # 5) Salva relatórios txt rápidos
    with open(out_dir / f"{tag}_report.txt", "w", encoding="utf-8") as f:
        f.write(f"[{tag}] K={K}\n")
        f.write("\n=== BASELINE ===\n")
        f.write(f"Acurácia: {acc_b:.4f}\n{rep_b}\n")
        f.write("\n=== cGAN ===\n")
        f.write(f"Acurácia: {acc_g:.4f}\n{rep_g}\n")
        f.write("\n=== GAN OUT ===\n")
        f.write(f"synthetic_csv: {gan_out['synthetic_csv']}\n")
        f.write(f"checkpoints: {gan_out['checkpoints']}\n")
        f.write(f"config: {gan_out['config']}\n")

    return {
        "baseline": {"acc": acc_b, "report": rep_b, "plot": str(out_dir / f"{tag}_baseline_k{K}.png")},
        "cgan":     {"acc": acc_g, "report": rep_g, "plot": str(out_dir / f"{tag}_cgan_k{K}.png")},
        "gan_out":  gan_out,
        "dir":      str(out_dir),
    }


## Varredura

In [ ]:
# carregar datasets e curvas teóricas
H_list, theoretical_list = load_all()

# configuração K e cGAN
K = 50  # como no artigo; região ótima ~30–60  
gan_cfg = dict(EPOCHS=100, BATCH_SIZE=80, LATENT_DIM=64, LR=2e-4, SEED=42)
samples_per_label = 500

results = {}
for h in (1, 2, 3):
    print(f"\n>>> Rodando varredura para H{h} ...")
    results[h] = run_one(
        H_list, theoretical_list,
        H_idx_test=h, K=K, gan_cfg=gan_cfg, samples_per_label=samples_per_label
    )

# resumo na tela
for h in (1, 2, 3):
    rb = results[h]["baseline"]["acc"]
    rg = results[h]["cgan"]["acc"]
    print(f"H{h}: baseline={rb:.4f} | cGAN={rg:.4f} | dir={results[h]['dir']}")
